In [5]:
import os
import json

import numpy as np
import pandas as pd

In [6]:
# runs = [
#     os.path.join('experiments', str(x)) for x in range(744, 824)
# ] # without +/- fix in Z_0 calculation

runs = [
    os.path.join('experiments', str(x)) for x in range(1154, 1194)
] # fixed +/- in Z_0 calculation

# runs = [
#     os.path.join('experiments', str(x)) for x in range(1417, 1457)
# ]

# assert len(runs) == 80, len(runs)

# assert len(runs) == 40, len(runs)

In [7]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [8]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))
    # assert info['use_cpu'] == False
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] == min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()
    
    df.loc[len(df)] = info

In [9]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to min. RL2E. (s)  \
Solution        w/ BC                                                
Hopf fibration  False             4.7 ± 0.1            226.2 ± 0.8   
                True            23.6 ± 14.6            228.5 ± 0.5   
Plane waves     False             7.5 ± 0.3            227.1 ± 2.2   
                True              7.3 ± 0.3            226.6 ± 3.0   
Radial waves    False            15.1 ± 0.7            229.3 ± 1.1   
                True            51.8 ± 11.6            227.4 ± 1.5   
Random solution False             7.1 ± 0.2            227.2 ± 1.2   
                True            54.5 ± 11.6            228.2 ± 1.3   

                            Min. RL2E. (%)  
Solution        w/ BC                       
Hopf fibration  False  2.23e-03 ± 1.50e-04  
                True   1.51e-02 ± 5.38e-03  
Plane waves     False  2.31e-03 ± 1.33e-04  
                True   4.48e-03 ± 8.07e-04  
Radial waves    False  7.73e-03 ± 2.75e-04  
                True   3.01e-02 ± 5.78e-03  
Random solution False  5.26e-03 ± 2.12e-04  
                True   2.43e-02 ± 4.17e-03

In [17]:
for column in df.columns:
    if 'time' in column or 'min' in column or 'max' in column or 'mu' in column or 'sigma' in column:
        continue
    try:
        print(column, df[column].unique().tolist())
    except:
        print(column, 'not possible')

input_dim [3]
output_dim [6]
log_freq [100]
verbose [False]
tol [0.0001]
atol [0.0001]
k [10]
model_class ['MaxwellSimple']
norm_inputs [False]
data_from_lattice [False]
n_val [10000]
init ['xavier normal']
noise_scale [0]
inner_lr [0.01]
inner_wd [5e-06]
outer_lr [0.01]
outer_wd [5e-05]
bilevel [False]
symlog [False]
early_stopping [False]
seed [1, 2, 3, 4, 5]
n_train [2000]
batch_size [2000]
width [10000]
activation ['tanh']
lr [0.05]
wd [5e-05]
use_cpu [False]
preload_data [False]
add_bc [True, False]
soln [1, 2, 3, 4]
scheduler ['cosine']
c [1.0]
param_count [200000]
soln_name ['Plane waves', 'Radial waves', 'Hopf fibration', 'Random solution']


In [18]:
import torch

In [21]:
x = torch.zeros((10,10))

In [31]:
torch.nn.init.xavier_normal_(x, gain=5)
x.mean()

tensor(-0.0224)